# Seed media — 61 photographs for brgen, amber and bsdports

**Runtime → Change runtime type → T4 GPU** before running.

Set `HF_TOKEN` in the sidebar (🔑) to a token that has accepted the
[FLUX.1-dev licence](https://huggingface.co/black-forest-labs/FLUX.1-dev).
A token without it downloads a 403 that surfaces hundreds of lines later
inside diffusers, so the setup cell checks for it before anything else.

Two models, because they cannot be one. Ragnhild's adapter says
`ss_base_model_version: sdxl_1.0` in its own metadata and its tensors are
`lora_te1_`/`lora_unet_` names that do not exist in a FLUX transformer — she
was trained on SDXL because a free T4 cannot hold FLUX.1-dev for training
either. Loading her into FLUX would attach nothing and render twelve
strangers.

1. **dating** — 12 frames, **SDXL** + her adapter at weight
   0.85. Released before FLUX loads, so 16 GB
   never has to hold both.
2. **scenes** — 32 frames, **FLUX.1-dev** nf4, no adapter
3. **amber** — 17 frames, **FLUX.1-dev** nf4, the one mannequin

The adapter is **not in the clone** and never will be: `STUDIO/lora/*/weights/`
is gitignored, because a 218 MB likeness does not belong in a public repo. Put
`ragnhild.safetensors` in `MyDrive/lora/ragnhild/` before running, or the
dating pass names every path it tried and skips.

Roughly 20–40 s a frame on a T4 at nf4, so about half an hour of GPU plus the
model download. Output goes to Drive; a disconnect costs the frames since the
last write and not the session.

Prompts are `STUDIO/lora/seed_media.yml`. Edits belong there — this notebook is
generated by `_toolkit/run_seed_media_colab.rb` and is overwritten.


In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("ok: HF_TOKEN from Colab secrets")
except Exception:
    import getpass
    os.environ["HF_TOKEN"] = getpass.getpass("HF token (FLUX.1-dev licence accepted): ")
assert os.environ.get("HF_TOKEN"), "no HF token"

# Fail here, not four hundred lines into a download. A token that has not
# accepted the licence gets 403 on the weights and the traceback names a
# missing file rather than a missing acceptance.
import urllib.request
req = urllib.request.Request(
    "https://huggingface.co/api/models/black-forest-labs/FLUX.1-dev",
    headers={"Authorization": f"Bearer {os.environ['HF_TOKEN']}"})
try:
    urllib.request.urlopen(req).read(1)
    print("ok: FLUX.1-dev reachable with this token")
except Exception as e:
    raise SystemExit(
        "FLUX.1-dev is not reachable with this token. Accept the licence at "
        "https://huggingface.co/black-forest-labs/FLUX.1-dev and rerun. " + str(e))


In [ ]:
# Frames go to Drive as they are made. A free session disconnects when idle and
# is capped near 12 h; writing at the end would make every disconnect cost the
# whole run.
import os
from google.colab import drive
drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/seed_media"
os.makedirs(OUT, exist_ok=True)
print("ok:", OUT)


In [ ]:
import subprocess, os
# bitsandbytes for nf4, peft for the adapter, sentencepiece for T5's tokenizer.
subprocess.run(
    "pip -q install -U diffusers transformers accelerate safetensors "
    "bitsandbytes peft sentencepiece protobuf",
    shell=True, check=True)

if not os.path.isdir("/content/pub4/.git"):
    subprocess.run(
        "git clone --depth 1 --branch main https://github.com/anon987654321/pub4.git /content/pub4",
        shell=True, check=True)
print("ok: toolkit and prompts at /content/pub4")

# Assert the GPU before anything expensive believes it has one.
#
# Without this the notebook installs, clones, downloads twenty gigabytes and
# then renders on the CPU, where a 12B transformer is minutes per step -- so
# the first sign of a missed Runtime -> T4 is a frame that never arrives.
# Colab also nags about an idle GPU while these setup cells run, which is
# correct and means nothing; this is the check that does mean something.
import torch
if not torch.cuda.is_available():
    raise SystemExit(
        "no CUDA device. Runtime -> Change runtime type -> T4 GPU, then rerun. "
        "Do not switch to a standard runtime: FLUX.1-dev on CPU is minutes per step.")
free, total = torch.cuda.mem_get_info()
print(f"ok: {torch.cuda.get_device_name(0)}, {total / 1e9:.1f} GB total, {free / 1e9:.1f} GB free")
if total < 14e9:
    print("warn: under 14 GB — nf4 fits a 16 GB T4 with little to spare")


In [ ]:
import gc, glob, os, yaml, torch

SPEC = "/content/pub4/STUDIO/lora/seed_media.yml"
spec = yaml.safe_load(open(SPEC))
DTYPE = torch.float16  # T4 is Turing (sm_75) and has no bf16 at all.

RATIOS = {"3:2": (1216, 832), "4:5": (896, 1120), "1:1": (1024, 1024), "4:3": (1152, 864)}

def size_for(ratio):
    return RATIOS.get(ratio or spec["meta"]["aspect_ratio"], (1024, 1024))

def render(pipe, key, prompt, ratio, seed, **kw):
    path = os.path.join(OUT, key + ".png")
    if os.path.exists(path):          # Resume is free; a disconnect is not.
        print("skip", key); return
    w, h = size_for(ratio)
    image = pipe(prompt=prompt, width=w, height=h,
                 generator=torch.Generator("cpu").manual_seed(seed), **kw).images[0]
    image.save(path)
    print("ok", key, f"{w}x{h}")

def teardown(pipe):
    del pipe
    gc.collect(); torch.cuda.empty_cache()

# ---------------------------------------------------------------- pass 1
# Dating, on SDXL, because that is the model Ragnhild's adapter was trained
# against -- its own metadata says ss_base_model_version: sdxl_1.0, and its
# tensors are lora_te1_/lora_unet_ names that do not exist in a FLUX
# transformer. Loading it into FLUX would not fail usefully; it would attach
# nothing and render twelve strangers.
#
# She was trained on SDXL because a free T4 cannot hold FLUX.1-dev for
# training either. Same constraint, one step earlier.
dating = spec["dating"]
SUBJECT = dating["lora"]
# Looked for by path first, then by search. The mount is authenticated as the
# operator, so a recursive walk of their own Drive is the reliable way to find
# a file they uploaded without having to be told where they put it.
#
# The share link is deliberately NOT hardcoded here. A Drive file ID in a
# public repo is the same disclosure as committing the weights: anyone reading
# this file could fetch her likeness. The mount needs no link.
ADAPTER_CANDIDATES = [
    f"/content/drive/MyDrive/lora/{SUBJECT}/{SUBJECT}.safetensors",
    f"/content/drive/MyDrive/lora/{SUBJECT}/weights/{SUBJECT}.safetensors",
    f"/content/drive/MyDrive/{SUBJECT}.safetensors",
]
ADAPTER_CANDIDATES += sorted(glob.glob(f"/content/drive/MyDrive/lora/{SUBJECT}/*.safetensors"))
ADAPTER = next((p for p in ADAPTER_CANDIDATES if os.path.exists(p)), None)

if not ADAPTER:
    print(f"not at the expected paths — searching MyDrive for *.safetensors …")
    found = glob.glob("/content/drive/MyDrive/**/*.safetensors", recursive=True)
    # The subject's name in the filename or its directory wins; failing that,
    # the largest file, since a subject adapter is hundreds of MB and a stray
    # embedding is kilobytes.
    named = [p for p in found if SUBJECT.lower() in p.lower()]
    pool = named or found
    if pool:
        ADAPTER = max(pool, key=os.path.getsize)
        print(f"found {len(found)} adapter(s); using {ADAPTER}")
        if not named:
            print(f"warn: none named '{SUBJECT}' — this is the largest one, check the face")

if ADAPTER:
    # Say which base it was trained against before spending a GPU on it. An
    # SDXL adapter in a FLUX pipeline attaches nothing and renders strangers,
    # which is a failure that produces files and looks like success.
    import json as _json
    with open(ADAPTER, "rb") as _f:
        _n = int.from_bytes(_f.read(8), "little")
        _meta = _json.loads(_f.read(_n)).get("__metadata__", {})
    _base = _meta.get("ss_base_model_version", "unknown")
    _steps = _meta.get("training_info", "")
    print(f"ok: adapter base={_base} {_steps}")
    if "sdxl" not in _base.lower():
        print(f"warn: base is {_base}, and this pass builds an SDXL pipeline.")
        print("warn: if she has been retrained on FLUX, render her in the FLUX pass instead.")

if ADAPTER:
    from diffusers import StableDiffusionXLPipeline, AutoencoderKL
    # The fp16-fix VAE, not SDXL's own: the original overflows in fp16 and
    # decodes to black. render_config.rb names the same repo for the same
    # reason.
    vae = AutoencoderKL.from_pretrained("madebyollin/sdxl-vae-fp16-fix", torch_dtype=DTYPE)
    sd = StableDiffusionXLPipeline.from_pretrained(
        "stabilityai/stable-diffusion-xl-base-1.0",
        vae=vae, torch_dtype=DTYPE, variant="fp16", use_safetensors=True)
    sd.enable_model_cpu_offload()
    sd.set_progress_bar_config(disable=True)
    sd.load_lora_weights(ADAPTER, adapter_name="subject")
    sd.set_adapters(["subject"], adapter_weights=[dating["lora_weight"]])

    env_path = f"/content/pub4/STUDIO/lora/{SUBJECT}/subject.env"
    trigger, descriptor = SUBJECT, ""
    if os.path.exists(env_path):
        for line in open(env_path):
            if line.startswith("TRIGGER="):
                trigger = line.split("=", 1)[1].strip().strip('"').strip("'")
            if line.startswith("DESCRIPTOR="):
                descriptor = line.split("=", 1)[1].strip().strip('"').strip("'")
    print("ok: SDXL + adapter loaded, trigger =", trigger)

    subject_clause = f"{trigger}, {descriptor}" if descriptor else trigger
    for i, (key, prompt) in enumerate(dating["profiles"].items()):
        render(sd, key, prompt.replace("TRIGGER", subject_clause),
               dating["aspect_ratio"], 1000 + i,
               guidance_scale=6.0, num_inference_steps=30)
    teardown(sd)
    print("ok: dating pass complete, SDXL released")
else:
    print("SKIPPED dating — no adapter found. Looked in:")
    for p in ADAPTER_CANDIDATES:
        print("   ", p)
    print("")
    print(f"  Put {SUBJECT}.safetensors in MyDrive/lora/{SUBJECT}/ and rerun.")
    print("  It is gitignored on purpose — a 218 MB likeness does not belong")
    print("  in a public repo — so the clone will never carry it.")

# ------------------------------------------------------- passes 2 and 3
# Scenes and garments on FLUX.1-dev, loaded only now: SDXL is already gone, so
# the 16 GB does not have to hold both.
from diffusers import FluxPipeline, BitsAndBytesConfig, FluxTransformer2DModel
from transformers import T5EncoderModel, BitsAndBytesConfig as TfBnb

MODEL = "black-forest-labs/FLUX.1-dev"
# Transformer AND T5 quantised, both necessary: nf4 puts the transformer near
# 6.5 GB, but T5-XXL is another 9 GB in fp16 and 16 GB will not hold both plus
# activations.
nf4 = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=DTYPE)
transformer = FluxTransformer2DModel.from_pretrained(
    MODEL, subfolder="transformer", quantization_config=nf4, torch_dtype=DTYPE)
text_encoder_2 = T5EncoderModel.from_pretrained(
    MODEL, subfolder="text_encoder_2",
    quantization_config=TfBnb(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                              bnb_4bit_compute_dtype=DTYPE),
    torch_dtype=DTYPE)
flux = FluxPipeline.from_pretrained(
    MODEL, transformer=transformer, text_encoder_2=text_encoder_2, torch_dtype=DTYPE)
flux.enable_model_cpu_offload()
flux.set_progress_bar_config(disable=True)
print("ok: FLUX.1-dev loaded nf4")

FLUX_KW = dict(guidance_scale=spec["meta"]["guidance"],
               num_inference_steps=spec["meta"]["steps"])

for i, (key, entry) in enumerate(spec["scenes"].items()):
    render(flux, key, entry["prompt"], entry.get("aspect_ratio"), 2000 + i, **FLUX_KW)

# The mannequin clause is prepended rather than repeated in each entry, so
# seventeen garments share one mannequin instead of seventeen slightly
# different ones.
amber = spec["amber"]
for i, (key, garment) in enumerate(amber["garments"].items()):
    render(flux, key, spec["mannequin"] + ", " + garment,
           amber["aspect_ratio"], 3000 + i, **FLUX_KW)

made = len([f for f in os.listdir(OUT) if f.endswith(".png")])
print("")
print(f"ok: {made} frame(s) in {OUT}")
print("next: download that folder, then on the Mac:")
print("  ruby STUDIO/lora/_toolkit/install_seed_media.rb <folder>")
